In [1]:
#transformer总体架构
#输入：文本序列
#输出：文本序列
#编码器部分
#解码器部分

#输入部分包含词嵌入层和位置编码层
#词嵌入层将输入的文本序列转换为向量表示
#位置编码层为输入的文本序列添加位置信息，使模型能够捕捉到序列中单词之间的相对位置关系

#编码器部分包含多个编码器层，每个编码器层由多头自注意力机制和前馈神经网络组成
#每个编码器层由两个子层组成：多头自注意力机制和前馈神经网络
#第一个子层是多头自注意力机制以及一个残差连接，它允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系，残差连接有助于缓解深层网络中的梯度消失问题
#第二个子层包括一个前馈神经网络和规范化层以及一个残差连接，前馈神经网络由两个线性变换和一个激活函数组成，规范化层用于稳定训练过程，残差连接有助于缓解深层网络中的梯度消失问题

#解码器部分也包含多个解码器层，每个编码器层由三个子层组成：多头自注意力机制、编码器-解码器注意力机制和前馈神经网络，第一个子层为带掩码的多头自注意力机制以及残差连接和规范化层，第二个子层为多头自注意力机制层以及残差连接和规范化层，第三个子层为前馈神经网络层以及残差连接和规范化层

In [2]:
#文本嵌入层
#文本嵌入层将输入的文本序列转换为向量表示
#注意：在文本嵌入时，经常在embedding后乘以一个缩放因子，通常是嵌入维度的平方根。这是因为在Transformer中，输入的嵌入向量会被送入多头自注意力机制中进行计算，而多头自注意力机制中的点积操作可能会导致数值不稳定。通过乘以缩放因子，可以使得输入的嵌入向量的数值范围更适合进行点积计算，从而提高模型的训练稳定性和性能。

#文本嵌入层实现
import torch.nn as nn
import torch
import math


embedding=nn.Embedding(100,10,padding_idx=0) #词表大小为100，嵌入维度为10，padding_idx=0表示将索引为0的词视为填充词，不进行训练
input=torch.tensor([[1,2,3,4,0],[5,6,7,8,0]],dtype=torch.long) #输入的文本序列，包含两个样本，每个样本包含5个词，其中索引为0的词表示填充词
output=embedding(input)
print(output)

class Embedding(nn.Module):
    def __init__(self,vocab_size,embedding_dim):
        super().__init__()
        self.vocab_size=vocab_size
        self.embedding_dim=embedding_dim
        self.embed=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
    def forward(self,x):
        x=self.embed(x)
        return x*math.sqrt(self.embedding_dim) #乘以缩放因子

embedding=Embedding(100,20)
output=embedding(input)
print(output)

tensor([[[-0.5248, -0.2763,  0.3342,  0.3082, -0.1407, -0.1500, -0.5779,
           0.7477, -0.5937,  0.0890],
         [ 0.2474, -0.3250, -1.3679,  2.1013,  0.3333,  1.1862, -0.4558,
           0.8027, -0.6127,  0.4027],
         [ 0.2506,  1.0776,  1.3605,  0.7018,  0.9556,  1.4848,  0.1340,
           1.0972,  1.1611, -0.2006],
         [ 0.6647,  0.1642, -0.3713, -0.6368, -0.7929,  0.3528, -1.6835,
           0.7051,  0.4883, -1.3468],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000]],

        [[-0.2757,  0.8082,  0.7662,  0.2624,  0.2871,  0.2488, -0.2171,
          -0.9703,  0.2005, -1.3188],
         [-0.7733,  0.3996,  0.2967,  0.3974, -0.1446,  2.1046,  0.0323,
          -0.5014, -0.5152, -0.6920],
         [-0.3149,  0.1188, -0.2522,  0.8734,  0.7087,  1.1924,  1.8441,
          -0.8766,  0.8139,  1.1575],
         [ 0.0373,  0.7378, -0.6066, -0.6524, -0.0408, -1.0213, -0.3603,
           0.2256, -0.5731, -1.8261],

In [3]:
#位置编码层
#位置编码层为输入的文本序列添加位置信息，使模型能够捕捉到序列中单词之间的相对位置关系，将索引转化为位置编码向量
#三角函数位置编码公式
#PE(pos,2i)=sin(pos/10000^(2i/d_model)) 奇数位使用正弦函数，偶数位使用余弦函数
#PE(pos,2i+1)=cos(pos/10000^(2i/d_model))
#其中pos表示单词在序列中的位置，i表示嵌入维度的索引，d_model表示嵌入维度的大小

#位置编码层的实现
class PositionalEncoding(nn.Module):
    def __init__(self,dropout,d_model,max_len):
        super().__init__()
        self.dropout=nn.Dropout(dropout) #dropout层用于防止过拟合

        pe=torch.zeros(max_len,d_model) #初始化pe矩阵，大小为(max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小
        position=torch.arange(0,max_len,dtype=torch.long).unsqueeze(1) #生成位置索引，大小为(max_len, 1)，其中max_len表示最大序列长度
        div_term=torch.exp(torch.arange(0,max_len,dtype=torch.long)*-(math.log(10000)/d_model)) #计算分母项，大小为(d_model/2)，其中d_model表示嵌入维度的大小

        #position和div_term相乘得到一个大小为(max_len, d_model/2)的矩阵，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

         #计算位置编码矩阵，使用三角函数公式，奇数位使用正弦函数，偶数位使用余弦函数
         #pe矩阵的大小为(max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)
        pe=pe.unsqueeze(0) #添加维度，与词嵌入矩阵维度匹配，大小为(1, max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

        self.register_buffer('pe',pe) #将pe矩阵注册为模型的缓冲区，使其在模型保存和加载时能够正确处理，并且在训练过程中不会被更新
    def forward(self,x):
        x=x+self.pe[:,x.size(1)]
        return self.dropout(x)



In [4]:
#掩码张量的介绍
#掩码张量是一个布尔类型的张量，用于指示模型在计算注意力权重时应该忽略哪些位置。掩码张量的形状通常为(batch_size, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度。掩码张量中的值为True表示对应位置应该被忽略，值为False表示对应位置应该被考虑。
#在Transformer模型中，掩码张量主要用于两种情况：填充掩码和未来掩码。填充掩码用于指示模型在计算注意力权重时应该忽略填充位置，未来掩码用于指示模型在计算注意力权重时应该忽略未来位置，以防止模型在训练过程中看到未来的信息。

#transformer中自注意力机制的实现
#自注意力机制允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系。自注意力机制的核心是计算注意力权重，这些权重表示输入序列中每个位置对其他位置的关注程度。注意力权重的计算通常使用点积操作，具体来说，给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V，然后计算注意力权重矩阵A，A=softmax(QK^T/sqrt(d_k))，其中d_k是键矩阵的维度。最后，通过将注意力权重矩阵A与值矩阵V相乘，得到自注意力机制的输出矩阵O，O=AV。自注意力机制的输出矩阵O包含了输入序列中每个位置对其他位置的关注信息，从而使模型能够捕捉到输入序列中的长距离依赖关系。

#自注意力机制中的Q,K,V矩阵的计算
#给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V。具体来说，给定一个输入序列的表示矩阵X，首先通过三个不同的线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V。线性变换的参数通常是可学习的权重矩阵W_Q、W_K和W_V，分别用于计算查询矩阵Q、键矩阵K和值矩阵V。

In [5]:
#生成掩码向量

#测试生成上三角矩阵
import numpy as np
print(np.triu([[1,1,1],[0,1,1],[0,0,1]],k=1)) #生成一个上三角矩阵，k=1表示主对角线以上的元素保留，主对角线及以下的元素为0
print(np.triu([[1,1,1],[0,1,1],[0,0,1]],k=0)) #生成一个上三角矩阵，k=0表示主对角线以上的元素保留，主对角线以下的元素为0

size=5
mask=torch.from_numpy(1-np.triu(np.ones((1,size,size)),k=1).astype(int)) #生成一个掩码举着呢，三维，np.triu生成一个上三角矩阵，k=1表示主对角线以上的元素保留，主对角线及以下的元素为0，np.ones((1,size,size))生成一个全1的三维矩阵，1-np.triu(...)将上三角矩阵中的1变为0，0变为1，最后使用torch.from_numpy将其转换为PyTorch张量
print(mask)

[[0 1 1]
 [0 0 1]
 [0 0 0]]
[[1 1 1]
 [0 1 1]
 [0 0 1]]
tensor([[[1, 0, 0, 0, 0],
         [1, 1, 0, 0, 0],
         [1, 1, 1, 0, 0],
         [1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1]]], dtype=torch.int32)


In [6]:
#sentence_mask：用于解码器中的未来掩码
#生成一个句子掩码，用于指示模型在计算注意力权重时应该忽略哪些位置。句子掩码通常用于指示模型在计算注意力权重时应该忽略填充位置，以防止模型在训练过程中看到填充的信息。

#例如，给定一个输入序列的表示矩阵X和一个对应的句子掩码mask，模型在计算注意力权重时会将mask中为True的位置对应的注意力权重设置为负无穷，从而使得这些位置在计算注意力权重时被忽略。


In [7]:
#sentence_mask演示
torch.manual_seed(42)
input=torch.randn(1,size,size)
mask=torch.from_numpy(1-np.triu(np.ones((1,size,size)),k=1).astype(int))
print(mask)
print(input)
input.masked_fill_(mask==0,float('-inf')) #使用掩码填充的方法将mask中为0的位置赋值为负无穷，最后使得这些位置在计算权重的时候被忽略

tensor([[[1, 0, 0, 0, 0],
         [1, 1, 0, 0, 0],
         [1, 1, 1, 0, 0],
         [1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1]]], dtype=torch.int32)
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
         [-1.2345, -0.0431, -1.6047, -0.7521, -0.6866],
         [-0.4934,  0.2415, -1.1109,  0.0915, -2.3169],
         [-0.2168, -1.3847, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]]])


tensor([[[ 1.9269,    -inf,    -inf,    -inf,    -inf],
         [-1.2345, -0.0431,    -inf,    -inf,    -inf],
         [-0.4934,  0.2415, -1.1109,    -inf,    -inf],
         [-0.2168, -1.3847, -0.3957,  0.8034,    -inf],
         [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]]])

In [8]:
#padding_mask的原理：用于编码器中的填充掩码
#生成一个填充掩码，用于指示模型在计算注意力权重时应该忽略哪些位置。填充掩码通常用于指示模型在计算注意力权重时应该忽略填充位置，以防止模型在训练过程中看到填充的信息。

#padding_mask的实现
a=torch.randn(size,size)
print(a!=0) #生成一个布尔类型的掩码张量，大小为(size, size)，其中size表示序列长度，a!=0表示将a中不等于0的位置对应的掩码值设置为True，等于0的位置对应的掩码值设置为False
a[a!=0]=1
print(a)

b=torch.tensor([[1,2,0,0],[4,5,0,0],[0,0,0,0],[0,0,0,0]])
padding_mask=(b!=0) #生成一个填充掩码，用于指示模型在计算注意力权重时应该忽略哪些位置，大小为(batch_size, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，b!=0表示将b中不等于0的位置对应的掩码值设置为True，等于0的位置对应的掩码值设置为False
print(padding_mask)


tensor([[True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True]])
tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])
tensor([[ True,  True, False, False],
        [ True,  True, False, False],
        [False, False, False, False],
        [False, False, False, False]])


In [9]:
#实现自注意力计算
import torch.nn.functional as F

class Selfattention(nn.Module): #实现自注意力机制的计算，输入为查询矩阵Q、键矩阵K和值矩阵V，以及可选的掩码张量mask和dropout概率dropout
    def __init__(self,q,k,v,mask=None,dropout=None): #q=k=v=position_x,自注意力机制
        super().__init__()
        self.q=q
        self.k=k
        self.v=v
        self.mask=mask
        self.dropout=dropout
    def forward(self):
        d_model=self.q.size(-1) #词嵌入维度
        scores=torch.matmul(self.q,self.k.traspose(-2,-1))/torch.sqrt(torch.tensor(d_model))  #计算注意力权重矩阵，使用点积操作，大小为(batch_size, seq_len, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，self.q.size(-1)表示查询矩阵的最后一个维度，即词嵌入维度
        weights=F.softmax(scores,dim=-1) #进行归一化，得到注意力权重矩阵，大小为(batch_size, seq_len, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，dim=-1表示在最后一个维度上进行softmax操作，即对每个查询位置的注意力权重进行归一化，使得它们的和为1

        if mask is not None:
            weights=weights.masked_fill_(mask==0,float('-inf')) #掩码操作，将mask中为0的位置对应的注意力权重设置为负无穷，最后使得这些位置在计算权重的时候被忽略

        if self.dropout is not None:
            weights=nn.Dropout(self.dropout)(weights)

        return torch.matmul(weights,self.v),weights


In [10]:
#多头注意力机制的理解
#多头注意力机制是Transformer模型中的一个重要组件，它允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系。多头注意力机制通过将查询矩阵Q、键矩阵K和值矩阵V分别映射到多个不同的子空间中，来实现对输入序列的多方面关注。具体来说，给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V，然后将这些矩阵分别划分为多个头，每个头对应一个不同的子空间。对于每个头，计算注意力权重矩阵A，并将其与值矩阵V相乘，得到每个头的输出矩阵O。最后，将所有头的输出矩阵O进行拼接，并通过线性变换得到多头注意力机制的最终输出矩阵。这种机制使得模型能够同时关注输入序列中的多个位置，从而更好地捕捉到输入序列中的长距离依赖关系。

#例如输入注意力计算之前，q=k=v=[2,4,512]，多头注意力机制将其映射到多个不同的子空间中，例如8个头，每个头对应一个64维的子空间，那么每个头的查询矩阵Q、键矩阵K和值矩阵V的大小分别为[2,4,64]。对于每个头，计算注意力权重矩阵A，并将其与值矩阵V相乘，得到每个头的输出矩阵O，大小为[2,4,64]。最后，将所有头的输出矩阵O进行拼接，并通过线性变换得到多头注意力机制的最终输出矩阵，大小为[2,4,512]。

In [11]:
#多头注意力机制的实现
class MultiheadAttention(nn.Module):
    def __init__(self,embedding_dim,heads,dropout):
        super().__init__()
        self.embedding_dim=embedding_dim
    
        
        self.heads=heads
        self.dropout=dropout

        assert embedding_dim%heads==0 #确保嵌入维度能够被头数整除
        #assert语句用于检查条件是否满足，如果条件不满足，则会抛出一个AssertionError异常，并显示指定的错误消息。在这里，assert语句用于确保嵌入维度能够被头数整除，以便每个头的维度能够正确计算。

        self.d_k=embedding_dim//heads #每个头的维度

        self.q_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将输入的表示矩阵映射到查询矩阵Q
        self.k_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将输入的表示矩阵映射到键矩阵K
        self.v_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将输入的表示矩阵映射到值矩阵V

        self.out_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将多头注意力机制的输出矩阵映射回原始维度

        self.attention_weights=None
        self.dropout=nn.Dropout(dropout)
    def forward(self,q,k,v,mask=None):

        batch_size=q.size(0)
        #进行线性投影
        q=self.q_linear(q)
        k=self.k_linear(k)
        v=self.v_linear(v)

        #进行多头切分
        q=q.view(batch_size,-1,self.heads,self.d_k)
        k=k.view(batch_size,-1,self.heads,self.d_k)
        v=v.view(batch_size,-1,self.heads,self.d_k) 

        #进行维度交换
        q=q.transpose(1,2)
        k=k.transpose(1,2)
        v=v.transpose(1,2)

        #进行自注意力计算
        scores=torch.matmul(q,k.transpose(-2,-1))/torch.sqrt(torch.tensor(self.d_k))

        #掩码处理
        if mask is not None:
            scores.masked_fill_(mask==0,float('-inf'))
        
        weights=F.softmax(scores,dim=-1)

        weights=self.dropout(weights)
        output=torch.matmul(weights,v) #进行注意力计算，得到多头注意力的初始输出矩阵，大小为(batch_size, heads, seq_len, d_k)，其中batch_size表示批次大小，heads表示头数，seq_len表示序列长度，d_k表示每个头的维度

        output=output.transpose(1,2) #将heads和seq_len进行维度交换，得到大小为(batch_size, seq_len, heads, d_k)的矩阵，其中batch_size表示批次大小，seq_len表示序列长度，heads表示头数，d_k表示每个头的维度
        output=output.contiguous().view(batch_size,-1,self.embedding_dim) #将最后两个维度相乘，得到大小为(batch_size, seq_len, embedding_dim)的矩阵，其中batch_size表示批次大小，seq_len表示序列长度，embedding_dim表示嵌入维度的大小

        output=self.out_linear(output) #进行输出维度的映射
        self.attention_weights=weights
        return output,weights

#测试
multihead=MultiheadAttention(embedding_dim=512,heads=8,dropout=0.1)
q=torch.randn(2,4,512)
k=q
v=q
mask=torch.from_numpy(1-np.triu(np.ones((2,4,4)),k=1).astype(int))
mask=mask.unsqueeze(1)
print(mask)
multi_output,weights=multihead(q,k,v,mask)
print(multi_output.shape)

print(weights.shape)

tensor([[[[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 1, 1]]],


        [[[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 1, 1]]]], dtype=torch.int32)
torch.Size([2, 4, 512])
torch.Size([2, 8, 4, 4])


In [12]:
#前馈全连接层
#前馈全连接层是Transformer模型中的一个重要组件，它由两个线性变换和一个激活函数组成。前馈全连接层的输入是来自多头注意力机制的输出矩阵，大小为(batch_size, seq_len, embedding_dim)，其中batch_size表示批次大小，seq_len表示序列长度，embedding_dim表示嵌入维度的大小。前馈全连接层首先通过一个线性变换将输入矩阵映射到一个更高维度的空间中，通常是4倍的嵌入维度，即hidden_dim=4*embedding_dim。然后，通过一个激活函数（例如ReLU）对映射后的矩阵进行非线性变换，得到一个新的矩阵。最后，通过另一个线性变换将这个新的矩阵映射回原始的嵌入维度，得到前馈全连接层的输出矩阵，大小为(batch_size, seq_len, embedding_dim)。

# 前馈全连接层的作用是对多头注意力机制的输出进行进一步的处理和变换，以增强模型的表达能力和非线性特征。

class Feedforward(nn.Module):
    def __init__(self,d_model,d_ff,dropout):
        super().__init__()
        self.d_model=d_model #词向量维数
        self.d_ff=d_ff #全连接层的维数，通常是d_model的四倍
        self.dropout=nn.Dropout(dropout) #dropout用于随机失活
        self.fc1=nn.Linear(self.d_model,self.d_ff)
        self.fc2=nn.Linear(self.d_ff,self.d_model) #最终结果又映射成原始的维度
    def forward(self,x):
        x=self.fc1(x)
        x=F.relu(x) #激活函数
        x=self.dropout(x)
        x=self.fc2(x)
        return x

#测试
feedback=Feedforward(d_model=512,d_ff=2048,dropout=0.1)
input=multi_output
feed=feedback(input)
print(feed)

print(feed.shape)



tensor([[[ 0.0043, -0.0082, -0.0563,  ..., -0.0943,  0.0725,  0.0785],
         [ 0.1360,  0.0641, -0.0597,  ..., -0.0512, -0.0270,  0.0379],
         [-0.0072,  0.0947,  0.0012,  ...,  0.0483,  0.0097, -0.0164],
         [ 0.0362,  0.0506, -0.0196,  ...,  0.0067,  0.0337,  0.0340]],

        [[ 0.1485,  0.0874, -0.0427,  ..., -0.0175, -0.0225,  0.1304],
         [-0.0373,  0.0954, -0.0694,  ...,  0.0330, -0.0096,  0.0566],
         [ 0.0164,  0.0583, -0.1032,  ...,  0.0098, -0.0561,  0.0890],
         [ 0.0720,  0.0175, -0.0874,  ...,  0.0006, -0.0015,  0.0348]]],
       grad_fn=<ViewBackward0>)
torch.Size([2, 4, 512])


In [13]:
#规范化层
#规范化层是Transformer模型中的一个重要组件，它用于稳定训练过程并加速模型的收敛。规范化层通过对输入数据进行归一化处理，使得输入数据的分布更加稳定，从而有助于缓解梯度消失和梯度爆炸问题。规范化层通常使用Layer Normalization（层归一化）或Batch Normalization（批归一化）来实现。

#Layernormalization是对每个样本的特征进行归一化处理，计算每个样本的均值和标准差，并使用这些统计量来对输入数据进行归一化。Layer Normalization在Transformer模型中更常用，因为它不依赖于批次大小，可以更好地适应不同的输入序列长度。

class Layernormal(nn.Module):
    def __init__(self,features,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.weight=nn.Parameter(torch.ones(features)) #可训练的参数，对归一化之后的数据进行缩放，作为权重使用
        self.bias=nn.Parameter(torch.zeros(features)) #可训练的参数，对归一化的数据进行平移，作为偏置使用
    def forward(self,x):
        mean=x.mean(dim=-1,keepdim=True) #求解平均值
        var=x.var(dim=-1,keepdim=True) #求解方差
        x=(x-mean)/torch.sqrt(var+self.eps) #归一化处理
        return self.weight*x+self.bias #对归一化之后的数据进行缩放和平移，得到输出，这样是为了避免归一化之后的数据过于平坦，失去表达能力，通过引入可训练的权重和偏置，可以让模型在训练过程中学习到适合当前任务的缩放和平移参数，从而增强模型的表达能力。
#测试
layernorm=Layernormal(512)
input=feed
norm_output=layernorm(input)
print(norm_output)

tensor([[[ 0.0022, -0.1299, -0.6392,  ..., -1.0427,  0.7249,  0.7888],
         [ 2.0833,  0.9426, -1.0206,  ..., -0.8862, -0.5015,  0.5280],
         [-0.1410,  1.9727,  0.0333,  ...,  1.0112,  0.2097, -0.3311],
         [ 0.7802,  1.0948, -0.4408,  ...,  0.1340,  0.7252,  0.7320]],

        [[ 1.7316,  1.0042, -0.5441,  ..., -0.2444, -0.3045,  1.5162],
         [-0.6048,  1.4213, -1.0950,  ...,  0.4693, -0.1821,  0.8290],
         [ 0.2780,  1.0435, -1.9101,  ...,  0.1568, -1.0489,  1.6047],
         [ 1.5760,  0.3586, -1.9836,  ..., -0.0198, -0.0654,  0.7446]]],
       grad_fn=<AddBackward0>)


In [14]:
#子层连接结构的实现

#子层连接结构的作用：
#子层连接结构是Transformer模型中的一个重要组件，它用于连接多头注意力机制和前馈全连接层等子层，并通过残差连接和规范化层来稳定训练过程。子层连接结构的输入是来自前一个子层的输出矩阵，大小为(batch_size, seq_len, embedding_dim)，其中batch_size表示批次大小，seq_len表示序列长度，embedding_dim表示嵌入维度的大小。子层连接结构首先将输入矩阵传递给当前子层进行处理，得到一个新的矩阵。然后，通过一个残差连接将这个新的矩阵与输入矩阵相加，得到一个新的矩阵。最后，通过一个规范化层对这个新的矩阵进行归一化处理，得到子层连接结构的输出矩阵，大小为(batch_size, seq_len, embedding_dim)。
class Sublayerconnection(nn.Module):
    def __init__(self,size,dropout):
        super().__init__()
        self.norm=Layernormal(size,dropout)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x,sublayer):
        return x+self.dropout(self.norm(sublayer(x))) #post_norm结构，先进行子层处理，再进行规范化和dropout，最后与输入矩阵x相加得到输出矩阵

        #pre_norm结构，先进行规范化和dropout，再进行子层处理，最后与输入矩阵x相加得到输出矩阵
        #return x+self.dropout((sublayer(self.norm(x))))

        #sublayer是一个函数，表示当前子层的处理逻辑，例如多头注意力机制或前馈全连接层，sublayer(x)表示将输入矩阵x传递给当前子层进行处理，得到一个新的矩阵，然后通过规范化层进行归一化处理，并通过dropout进行随机失活，最后将这个新的矩阵与输入矩阵x相加，得到子层连接结构的输出矩阵。

#意思就是先将输入矩阵x传递给当前子层进行处理，得到一个新的矩阵，然后通过规范化层进行归一化处理，并通过dropout进行随机失活，最后将这个新的矩阵与输入矩阵x相加，得到子层连接结构的输出矩阵。这样做的目的是为了稳定训练过程并增强模型的表达能力，通过残差连接可以缓解梯度消失问题，而规范化层可以使得输入数据的分布更加稳定，从而加速模型的收敛，sublayer可以指多头注意力层或者前馈神经网络层等子层




